# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ismayilysfli/FlyRank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Signal 1**— March impressions / volume: OPPOSITE
Higher-volume pages were less likely to show a >20% April decline: the decline rate fell from 53.85% in the 100–499 bucket to 44.55% for pages with 10,000+ impressions. I therefore will not treat high volume as evidence of higher decline risk. I will use volume only as an importance filter so the queue focuses on pages with enough exposure to matter.

**Signal 2** — March momentum: CONFIRMED
Pages already losing impressions during March were much more likely to decline in April. The future decline rate was 77.12% for the strong-decline bucket versus 36.78% for strong-growth pages. This supports using recent negative momentum as the main signal in the baseline rule.
Rule: Flag pages with at least 500 March impressions and a strong March decline of at least 30%. Rank flagged pages higher when the decline is more severe and the page has more search exposure.
Reason code: strong_decline_with_volume
Action: review_for_refresh
This is a review recommendation, not evidence that refreshing the page will cause recovery.

In [1]:
%pip -q install duckdb

import duckdb
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'"
    f")"
)

APRIL = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/month=2026-04/*.parquet'"
    f")"
)

print("Warehouse connected.")

Warehouse connected.


In [2]:
page_data = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS march_impressions,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                    THEN gsc_impressions ELSE 0
                END
            ) AS early_15_impressions,

            SUM(
                CASE
                    WHEN report_date BETWEEN DATE '2026-03-17' AND DATE '2026-03-31'
                    THEN gsc_impressions ELSE 0
                END
            ) AS late_15_impressions

        FROM {MARCH}
        WHERE gsc_data_available IS TRUE

        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    ),

    april AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS april_impressions

        FROM {APRIL}
        WHERE gsc_data_available IS TRUE

        GROUP BY 1, 2
    )

    SELECT
        m.*,
        a.april_impressions,

        CASE
            WHEN early_15_impressions > 0
            THEN 100.0 *
                 (late_15_impressions - early_15_impressions)
                 / early_15_impressions
            ELSE NULL
        END AS march_change_pct,

        CASE
            WHEN a.april_impressions < 0.8 * m.march_impressions
            THEN 1 ELSE 0
        END AS april_decline

    FROM march m
    INNER JOIN april a
        USING (client_hash_id, content_hash_id)
""").df()

print(f"Pages available for signal checks: {len(page_data):,}")
print(f"April decline rate: {page_data['april_decline'].mean():.3f}")

page_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages available for signal checks: 100,893
April decline rate: 0.515


,client_hash_id,content_hash_id,march_impressions,early_15_impressions,late_15_impressions,april_impressions,march_change_pct,april_decline
0,client_62f4a7e64f5e0096,content_76c1f31e2b38f054,747.0,590.0,147.0,249.0,-75.084746,1
1,client_62f4a7e64f5e0096,content_ffc5ab4b34aab1f8,501.0,358.0,135.0,281.0,-62.290503,1
2,client_62f4a7e64f5e0096,content_9739856fc83dc1ca,2893.0,1341.0,1473.0,722.0,9.843400,1
3,client_62f4a7e64f5e0096,content_3d1dc691a3502105,6955.0,2274.0,4601.0,3068.0,102.330695,1
4,client_62f4a7e64f5e0096,content_9d28af4f99c5e67b,6789.0,2440.0,4139.0,3896.0,69.631148,1


In [3]:
# Signal 1 — March search volume

page_data["volume_bucket"] = pd.cut(
    page_data["march_impressions"],
    bins=[99, 499, 1999, 9999, float("inf")],
    labels=["100-499", "500-1,999", "2,000-9,999", "10,000+"]
)

volume_check = (
    page_data
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        april_decline_rate=("april_decline", "mean")
    )
    .reset_index()
)

volume_check["april_decline_rate"] *= 100

print("Signal 1 — March impressions / volume")
volume_check

Signal 1 — March impressions / volume


,volume_bucket,n,april_decline_rate
0,100-499,39047,53.845366
1,"500-1,999",31983,52.731138
2,"2,000-9,999",23986,47.677812
3,"10,000+",5877,44.546537


In [4]:
# Signal 2 — March momentum

page_data["momentum_bucket"] = pd.cut(
    page_data["march_change_pct"],
    bins=[float("-inf"), -30, -10, 10, 30, float("inf")],
    labels=[
        "strong decline",
        "mild decline",
        "stable",
        "mild growth",
        "strong growth"
    ]
)

momentum_check = (
    page_data
    .dropna(subset=["momentum_bucket"])
    .groupby("momentum_bucket", observed=True)
    .agg(
        n=("content_hash_id", "size"),
        april_decline_rate=("april_decline", "mean")
    )
    .reset_index()
)

momentum_check["april_decline_rate"] *= 100

print("Signal 2 — March momentum")
momentum_check

Signal 2 — March momentum


,momentum_bucket,n,april_decline_rate
0,strong decline,21473,77.120104
1,mild decline,14405,62.124262
2,stable,14496,53.221578
3,mild growth,11646,46.445131
4,strong growth,34114,36.782553


## 2. Build the ranked queue (writes the CSV)

use one transparent baseline rule: pages qualify when they have at least 500 March impressions and their late-March impressions are at least 30% lower than early March. Among qualifying pages, the score increases with both decline severity and search exposure. Volume is used as an importance signal rather than as evidence that decline is more likely.
The score is normalized to a 0–100 scale. Every flagged page receives the reason code strong_decline_with_volume and the action label review_for_refresh. The resulting ranked queue is written to work/outputs/baseline_action_score.csv. Only March information is used in the score.

In [5]:
import numpy as np
march_only = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
                THEN gsc_impressions ELSE 0
            END
        ) AS early_15_impressions,

        SUM(
            CASE
                WHEN report_date BETWEEN DATE '2026-03-17' AND DATE '2026-03-31'
                THEN gsc_impressions ELSE 0
            END
        ) AS late_15_impressions

    FROM {MARCH}
    WHERE gsc_data_available IS TRUE

    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
""").df()

march_only["march_change_pct"] = (
    100
    * (march_only["late_15_impressions"] - march_only["early_15_impressions"])
    / march_only["early_15_impressions"].replace(0, np.nan)
)

In [6]:
import numpy as np
from pathlib import Path

# March-only baseline candidates
queue = march_only[
    (march_only["march_impressions"] >= 500)
    & (march_only["march_change_pct"] <= -30)
].copy()

# More negative momentum = more severe decline
queue["decline_severity"] = (
    -queue["march_change_pct"]
).clip(lower=0, upper=100)

# Volume is business importance, not a decline predictor
queue["volume_weight"] = np.log1p(queue["march_impressions"])

queue["raw_score"] = (
    queue["decline_severity"] * queue["volume_weight"]
)

queue["baseline_action_score"] = (
    100 * queue["raw_score"] / queue["raw_score"].max()
)

queue["reason_code"] = "strong_decline_with_volume"
queue["action_label"] = "review_for_refresh"

queue = (
    queue
    .sort_values("baseline_action_score", ascending=False)
    .reset_index(drop=True)
)

queue["baseline_rank"] = queue.index + 1

# Keep only March information in the exported action queue
output_columns = [
    "client_hash_id",
    "content_hash_id",
    "baseline_rank",
    "baseline_action_score",
    "march_impressions",
    "early_15_impressions",
    "late_15_impressions",
    "march_change_pct",
    "reason_code",
    "action_label",
]

queue_output = queue[output_columns]

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
queue_output.to_csv(output_path, index=False)

print(f"Flagged pages: {len(queue_output):,}")
print(f"Wrote: {output_path}")

queue_output.head(10)

Flagged pages: 12,524
Wrote: work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,baseline_rank,baseline_action_score,march_impressions,early_15_impressions,late_15_impressions,march_change_pct,reason_code,action_label
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,1,100.000000,83834.0,83772.0,58.0,-99.930764,strong_decline_with_volume,review_for_refresh
1,client_62f4a7e64f5e0096,content_945d6ff91386c817,2,89.274160,58278.0,49314.0,3862.0,-92.168553,strong_decline_with_volume,review_for_refresh
2,client_23a62021009f63c4,content_afa44be39cea94ca,3,84.070203,25704.0,23649.0,1468.0,-93.792549,strong_decline_with_volume,review_for_refresh
3,client_23a62021009f63c4,content_65c75874a23fca87,4,83.994265,64935.0,55680.0,7867.0,-85.871049,strong_decline_with_volume,review_for_refresh
4,client_62f4a7e64f5e0096,content_1642f339bd6e7c8d,5,83.818746,65330.0,52378.0,7519.0,-85.644736,strong_decline_with_volume,review_for_refresh
5,client_65de48885f4ef01b,content_62673eea26c31c17,6,80.984630,57720.0,49386.0,8058.0,-83.683635,strong_decline_with_volume,review_for_refresh
6,client_23a62021009f63c4,content_580e7d0863f1bc6d,7,80.461131,37967.0,29813.0,4041.0,-86.445510,strong_decline_with_volume,review_for_refresh
7,client_73cda7b4e4f265ea,content_09b05ffb4d7f6c8d,8,78.845699,50007.0,41590.0,7256.0,-82.553498,strong_decline_with_volume,review_for_refresh
8,client_62f4a7e64f5e0096,content_6a56a2183dc01691,9,78.806532,9461.0,9226.0,229.0,-97.517884,strong_decline_with_volume,review_for_refresh
9,client_73cda7b4e4f265ea,content_c9f840183215651b,10,78.274692,21519.0,18421.0,2048.0,-88.882254,strong_decline_with_volume,review_for_refresh


## 3. Top-10 review

I reviewed the top 10 manually rather than treating the score as an automatic refresh decision. All ten are flagged because they combine meaningful March visibility with a severe negative March momentum signal.
The action for each is review_for_refresh, but the recommendation could still be wrong if the decline is caused by seasonality, a temporary demand spike, consolidation into another page, technical/indexing issues, changes in search demand, or other factors that this baseline does not observe. The table below records the action, why each page appears in the queue, and one reason the recommendation could be wrong.

In [7]:
top10 = queue_output.head(10).copy()

wrong_reasons = [
    "The drop could be seasonal or caused by a temporary early-March demand spike.",
    "A related page may have absorbed the search demand instead of this page genuinely weakening.",
    "The decline could reflect a tracking or Search Console coverage issue.",
    "The first half of March may have contained an unusual short-term traffic spike.",
    "Demand for the topic may simply have fallen rather than the content becoming weaker.",
    "The page may have been intentionally redirected, consolidated, or deprioritized.",
    "A technical indexing issue rather than content quality could explain the decline.",
    "Search demand may have shifted to another query or related page.",
    "SERP changes could reduce impressions without a refresh being the right response.",
    "The decline may be temporary and could recover without any content change.",
]

review_rows = []

for i, row in top10.iterrows():
    review_rows.append({
        "rank": int(row["baseline_rank"]),
        "action": row["action_label"],
        "why_it_is_here": (
            f"{row['march_impressions']:,.0f} March impressions and "
            f"{row['march_change_pct']:.1f}% late-March change."
        ),
        "what_would_make_it_wrong": wrong_reasons[i],
    })

top10_review = pd.DataFrame(review_rows)

top10_review

,rank,action,why_it_is_here,what_would_make_it_wrong
0,1,review_for_refresh,"83,834 March impressions and -99.9% late-March...",The drop could be seasonal or caused by a temp...
1,2,review_for_refresh,"58,278 March impressions and -92.2% late-March...",A related page may have absorbed the search de...
2,3,review_for_refresh,"25,704 March impressions and -93.8% late-March...",The decline could reflect a tracking or Search...
3,4,review_for_refresh,"64,935 March impressions and -85.9% late-March...",The first half of March may have contained an ...
4,5,review_for_refresh,"65,330 March impressions and -85.6% late-March...",Demand for the topic may simply have fallen ra...
5,6,review_for_refresh,"57,720 March impressions and -83.7% late-March...",The page may have been intentionally redirecte...
6,7,review_for_refresh,"37,967 March impressions and -86.4% late-March...",A technical indexing issue rather than content...
7,8,review_for_refresh,"50,007 March impressions and -82.6% late-March...",Search demand may have shifted to another quer...
8,9,review_for_refresh,"9,461 March impressions and -97.5% late-March ...",SERP changes could reduce impressions without ...
9,10,review_for_refresh,"21,519 March impressions and -88.9% late-March...",The decline may be temporary and could recover...


## 4. Weak picks + leakage check

The highest-ranked pages have strong evidence under this baseline because they combine substantial March visibility with severe negative March momentum. However, the rule cannot distinguish true content decay from seasonality, demand shifts, consolidation into another page, technical search issues, or short-lived spikes. These are therefore review candidates, not automatic refresh decisions.
The baseline uses only March information. april_impressions and april_decline were used to audit the signals before the rule was chosen, but neither is included in the score or exported action queue. No product decision flags are used.

In [8]:
future_or_label_columns = {
    "april_impressions",
    "april_decline",
}

leaked_columns = future_or_label_columns.intersection(queue_output.columns)

print("Future/label-derived columns in exported queue:", leaked_columns)
print("Leakage check passed:", len(leaked_columns) == 0)

Future/label-derived columns in exported queue: set()
Leakage check passed: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.